# Is your hometown getting hotter?

**Terrain starter project · Python · NOAA Climate Data**

Download daily temperature records for a U.S. weather station, compute annual averages, and estimate a long-run warming trend.

**You will build:** a temperature trend chart with warming rate (°F per decade).

**Tip:** Change `STATION_ID` and `CITY_LABEL` below to analyze your hometown. Default is Los Angeles International Airport (LAX).

## 1. Setup

In [ ]:
from io import StringIO
from pathlib import Path
import pandas as pd
import requests
import matplotlib.pyplot as plt
import numpy as np

STATION_ID = "USW00023174"   # LAX — find others at https://www.ncei.noaa.gov/cdo-web/
CITY_LABEL = "Los Angeles (LAX)"
START_DATE = "1980-01-01"
END_DATE = "2024-12-31"
USE_BUNDLED_SAMPLE = False   # set True for offline demo using notebooks/data/

## 2. Download daily TMAX

In [ ]:
def fetch_noaa_daily(station, start, end):
    url = "https://www.ncei.noaa.gov/access/services/data/v1"
    params = {
        "dataset": "daily-summaries",
        "stations": station,
        "startDate": start,
        "endDate": end,
        "dataTypes": "TMAX",
        "format": "csv",
    }
    resp = requests.get(url, params=params, timeout=120)
    resp.raise_for_status()
    return pd.read_csv(StringIO(resp.text))

sample_path = Path("data/noaa_lax_tmax_daily.csv")
if not sample_path.exists():
    sample_path = Path("notebooks/data/noaa_lax_tmax_daily.csv")

if USE_BUNDLED_SAMPLE and sample_path.exists():
    daily = pd.read_csv(sample_path)
    print("Using bundled sample CSV")
else:
    daily = fetch_noaa_daily(STATION_ID, START_DATE, END_DATE)
    print(f"Downloaded {len(daily):,} daily rows for {STATION_ID}")

daily.head()

## 3. Clean and aggregate by year

In [ ]:
# NOAA stores TMAX in tenths of °C
work = daily.copy()
work["DATE"] = pd.to_datetime(work["DATE"])
work["TMAX_C"] = pd.to_numeric(work["TMAX"], errors="coerce") / 10.0
work = work.dropna(subset=["TMAX_C"])
work["TMAX_F"] = work["TMAX_C"] * 9 / 5 + 32

annual = (
    work.groupby(work["DATE"].dt.year, as_index=False)["TMAX_F"]
        .mean()
        .rename(columns={"DATE": "year", "TMAX_F": "mean_tmax_f"})
)
annual.head()

## 4. Fit a trend line

In [ ]:
x = annual["year"].to_numpy()
y = annual["mean_tmax_f"].to_numpy()
slope, intercept = np.polyfit(x, y, 1)
warming_per_decade = slope * 10

print(f"Mean annual TMAX: {y.mean():.1f} °F")
print(f"Trend: {warming_per_decade:+.2f} °F per decade ({START_DATE[:4]}–{END_DATE[:4]})")

## 5. Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(annual["year"], annual["mean_tmax_f"], s=18, alpha=0.75, color="#5B8BB0", label="Annual mean TMAX")
trend = intercept + slope * x
ax.plot(x, trend, color="#C45C5C", linewidth=2,
        label=f"Trend: {warming_per_decade:+.2f} °F/decade")
ax.set_title(f"Is {CITY_LABEL} getting hotter?")
ax.set_xlabel("Year")
ax.set_ylabel("Mean daily maximum temp (°F)")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Write your takeaway

1. Is the trend positive for your station? By how much per decade?
2. Cite the station ID and record length in one sentence.
3. Why is one weather station not the same as a whole city's climate?

**Next:** See the [NOAA explainer](../explainer.html?id=noaa-climate) or try the [air quality map](la_air_quality_map.ipynb).